In [0]:
# ── Imports ─────────────────────────────────────────────
import plotly.graph_objects as go
import pandas as pd

# ── Create dropdown (only once) ─────────────────────────
coin_ids = [row['coin_id'] for row in spark.sql("""
    SELECT DISTINCT coin_id 
    FROM crypto_db.gold_ohlc_history 
    ORDER BY coin_id
""").collect()]

dbutils.widgets.dropdown("coin", coin_ids[0], coin_ids, "Select Coin")

# ── Get selected value ──────────────────────────────────
selected_coin = dbutils.widgets.get("coin")

# ── Fetch data ──────────────────────────────────────────
df = spark.sql(f"""
    SELECT candle_date, open, high, low, close
    FROM crypto_db.gold_ohlc_history
    WHERE coin_id = '{selected_coin}'
    ORDER BY candle_date
""").toPandas()

# ── Handle empty case ───────────────────────────────────
if df.empty:
    print(f"No data available for {selected_coin}")
else:
    df["candle_date"] = pd.to_datetime(df["candle_date"])

    # Format label
    coin_label = selected_coin.replace("-", " ").title()

    # ── Build chart ─────────────────────────────────────
    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=df["candle_date"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        increasing_line_color="#10B981",
        increasing_fillcolor="#10B981",
        decreasing_line_color="#EF4444",
        decreasing_fillcolor="#EF4444",
        name=coin_label
    ))

    # ── Layout ─────────────────────────────────────────
    fig.update_layout(
        title=f"{coin_label} — OHLC Candlestick Chart",
        width=1200,
        height=600,
        plot_bgcolor="#1B2028",
        paper_bgcolor="#1B2028",
        font=dict(color="white"),
        yaxis=dict(
            title="Price (USD)",
            tickprefix="$",
            gridcolor="#2D3748"
        ),
        xaxis=dict(
            title="Date",
            gridcolor="#2D3748",
            rangeslider=dict(visible=False)
        )
    )

    print(f"{coin_label} — {len(df)} candles")
    fig.show()
import plotly.graph_objects as go
import pandas as pd

# Dropdown widget for coin selection
coin_ids = (
    spark.sql("SELECT DISTINCT coin_id FROM crypto_db.gold_ohlc_history ORDER BY coin_id")
    .toPandas()['coin_id']
    .tolist()
)
dbutils.widgets.dropdown('coin', 'bitcoin', coin_ids, 'Select Coin')
display(dbutils.widgets)  # Display the dropdown widget

selected_coin = dbutils.widgets.get('coin')

# Query OHLC data for selected coin
df = (
    spark.sql(f"""
        SELECT candle_date, open, high, low, close
        FROM crypto_db.gold_ohlc_history
        WHERE coin_id = '{selected_coin}'
        ORDER BY candle_date ASC
    """)
    .toPandas()
)
df['candle_date'] = pd.to_datetime(df['candle_date'])

# Build candlestick chart
coin_label = selected_coin.replace('-', ' ').title()

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df['candle_date'],
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    increasing_line_color='#10B981',
    increasing_fillcolor='#10B981',
    decreasing_line_color='#EF4444',
    decreasing_fillcolor='#EF4444',
    name='OHLC'
))

fig.update_layout(
    title=f'Select Coin: {coin_label} OHLC Candlestick Chart',
    width=1200,
    height=600,
    plot_bgcolor='#1B2028',
    paper_bgcolor='#1B2028',
    font=dict(color='white'),
    yaxis=dict(
        tickprefix='$',
        tickformat=',',
        gridcolor='#2D3748',
    ),
    xaxis=dict(
        gridcolor='#2D3748',
        rangeslider=dict(visible=False),
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
    ),
)

print(f'{coin_label} — {len(df)} candles')
fig.show()
